# PySpark concepts, end to end

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-concepts-end-to-end.ipynb)

You own a daily orders pipeline. Day one includes malformed records and repeated order updates. The next delivery includes a late refund, a stale update, and new orders. Finance expects the revenue table to reflect each event once, even when the scheduler retries the delivery.

Follow the data from raw files to typed records and quarantine, then to one current row per order and a daily revenue report. Inspect the expected counts at each boundary. Replay the second delivery and verify that the logical results match and that an untouched silver partition keeps its files.

This lab uses a single local writer and assumes that an order's business date stays fixed. The platform notes explain what needs to change before this becomes a production deployment.

**Prerequisites:** PySpark lessons [1](pyspark-01-how-spark-runs.ipynb) to [10](pyspark-10-batch-pipelines.ipynb); [13](pyspark-13-performance-tuning.ipynb) for the performance stage.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## The pipeline at a glance

| Stage | What it does | Module that teaches it |
| --- | --- | --- |
| 0. Session and landing zone | Local session, AQE on, raw files written to a temp folder | [PySpark 1](pyspark-01-how-spark-runs.ipynb) |
| 1. Bronze | Explicit schemas, `PERMISSIVE` reads, a quarantine for bad rows | [PySpark 4](pyspark-04-reading-and-writing-data.ipynb), [PySpark 10](pyspark-10-batch-pipelines.ipynb) |
| 2. Silver | Trim and normalise, latest-row dedupe with a window, `explode_outer` on nested events | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb), [PySpark 7](pyspark-07-aggregations-and-windows.ipynb), [PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb) |
| 3. Gold | Broadcast join to the customer dimension, daily revenue, per-customer ranking | [PySpark 6](pyspark-06-joins.ipynb), [PySpark 7](pyspark-07-aggregations-and-windows.ipynb) |
| 4. Incremental re-run | Dynamic partition overwrite of only the touched dates, run twice | [PySpark 10](pyspark-10-batch-pipelines.ipynb) |
| 5. Data quality | Uniqueness, nulls, ranges, referential integrity, reconciliation | [PySpark 10](pyspark-10-batch-pipelines.ipynb) |
| 6. Performance check | Broadcast vs sort-merge plan, partition pruning, one file per partition | [PySpark 12](pyspark-12-internals-and-monitoring.ipynb), [PySpark 13](pyspark-13-performance-tuning.ipynb) |

Every stage ends by showing its result, with notes on what to look for. Stage 5 adds a real
quality gate: the pipeline refuses to publish when a hard rule breaks. Streaming is out of scope here: the same bronze/silver/gold shape on a stream
is covered in [PySpark 11](pyspark-11-structured-streaming.ipynb).

## Stage 0 - Session and landing zone

Uses: building a local `SparkSession` and reading its config ([PySpark 1](pyspark-01-how-spark-runs.ipynb)). Two settings matter for the
pipeline: `partitionOverwriteMode=dynamic`, so an overwrite replaces only the partitions present
in the new data, and a small `shuffle.partitions` because the data is tiny.

In [ ]:
import os
import sys
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"  # every timestamp in the lab is UTC
time.tzset()

spark = (
    SparkSession.builder
    .appName("de-pyspark-capstone")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)

### Land the day-1 raw files

The source systems drop CSV and JSON-lines files into a landing folder. The day-1 files are
deliberately messy, the way real extracts are:

- **orders**: a status with stray case and whitespace, an order that was updated later the same
  day (two versions of order 3), an exact re-delivered duplicate (order 2), an unparseable amount,
  an unparseable timestamp, a missing key, a negative amount, and an order for a customer (`c9`)
  that is not in the customer extract.
- **customers**: untrimmed names and mixed-case country codes.
- **events**: a duplicated event, an empty and a missing `items` array, and one truncated JSON line.

In [ ]:
import tempfile

LAKE = tempfile.mkdtemp(prefix="de-pyspark-capstone-")
LANDING = os.path.join(LAKE, "landing")
os.makedirs(LANDING)

ORDERS_HEADER = "order_id,customer_id,amount,status,order_ts,updated_at"
RAW_FILES = {
    "orders_day1.csv": [
        ORDERS_HEADER,
        "1,c1,120.50,paid,2026-10-01 10:00:00,2026-10-01 10:00:00",
        "2,c2,80.00,PAID ,2026-10-01 11:30:00,2026-10-01 11:30:00",
        "3,c3,200.00,shipped,2026-10-02 09:15:00,2026-10-02 09:15:00",
        "3,c3,200.00,delivered,2026-10-02 09:15:00,2026-10-02 18:00:00",
        "4,c1,15.75,cancelled,2026-10-02 12:00:00,2026-10-02 12:00:00",
        "5,c9,60.00,paid,2026-10-03 08:45:00,2026-10-03 08:45:00",
        "6,c2,abc,paid,2026-10-03 16:20:00,2026-10-03 16:20:00",
        "7,c4,45.00,paid,not-a-date,not-a-date",
        ",c4,10.00,paid,2026-10-03 17:00:00,2026-10-03 17:00:00",
        "8,c4,-5.00,paid,2026-10-03 18:00:00,2026-10-03 18:00:00",
        "2,c2,80.00,PAID ,2026-10-01 11:30:00,2026-10-01 11:30:00",
    ],
    "customers.csv": [
        "customer_id,name,country",
        "c1, Ada ,us",
        "c2,Grace,GB",
        "c3,Linus,fi",
        "c4,Margaret,US",
    ],
    "events_day1.jsonl": [
        '{"event_id": "e1", "customer_id": "c1", "event_type": "view", "event_ts": "2026-10-01 09:55:00", "items": [{"sku": "A", "qty": 1}]}',
        '{"event_id": "e2", "customer_id": "c1", "event_type": "cart", "event_ts": "2026-10-01 09:58:00", "items": [{"sku": "A", "qty": 1}, {"sku": "B", "qty": 2}]}',
        '{"event_id": "e3", "customer_id": "c2", "event_type": "view", "event_ts": "2026-10-01 11:00:00", "items": []}',
        '{"event_id": "e4", "customer_id": "c3", "event_type": "view", "event_ts": "2026-10-02 09:00:00"}',
        '{"event_id": "e4", "customer_id": "c3", "event_type": "view", "event_ts": "2026-10-02 09:00:00"}',
        '{"event_id": "e5", "customer_id": "c4", "event_type": "cart", "event_ts": "2026-10-03 10:00:00", "items": [{"sku": "C", "qty": 3}]',
        '{"event_id": "e6", "customer_id": "c4", "event_type": "cart", "event_ts": "2026-10-03 10:05:00", "items": [{"sku": "C", "qty": 3}]}',
    ],
}


def land(name: str, rows: list[str]) -> str:
    """Write one raw extract into the landing folder and return its path."""
    path = os.path.join(LANDING, name)
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("\n".join(rows) + "\n")
    return path


landed = {name: land(name, rows) for name, rows in RAW_FILES.items()}
print("landed:", sorted(os.listdir(LANDING)))

## Stage 1 - Bronze: typed reads and a quarantine

Uses: explicit schemas and read modes ([PySpark 4](pyspark-04-reading-and-writing-data.ipynb)) and bad-record handling ([PySpark 10](pyspark-10-batch-pipelines.ipynb)).

The schemas are declared up front, never inferred, and money is `decimal(12,2)` so totals add up exactly. `PERMISSIVE` mode with a
`_corrupt_record` column keeps unparseable rows instead of failing the job or silently dropping
them. Rows that parse but break a business rule (no key, negative amount) are flagged too. Both
kinds go to a quarantine table with a reason, and only clean rows move on. Each row also carries
its source file name (from the `_metadata` column) and a batch id, which the dedupe and the
re-run rely on later.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    DecimalType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"
CORRUPT = "_corrupt_record"

ORDERS_SCHEMA = StructType([
    StructField("order_id", LongType()),
    StructField("customer_id", StringType()),
    StructField("amount", DecimalType(12, 2)),
    StructField("status", StringType()),
    StructField("order_ts", TimestampType()),
    StructField("updated_at", TimestampType()),
    StructField(CORRUPT, StringType()),
])

CUSTOMERS_SCHEMA = StructType([
    StructField("customer_id", StringType()),
    StructField("name", StringType()),
    StructField("country", StringType()),
])

ITEM = StructType([StructField("sku", StringType()), StructField("qty", IntegerType())])
EVENTS_SCHEMA = StructType([
    StructField("event_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_ts", TimestampType()),
    StructField("items", ArrayType(ITEM)),
    StructField(CORRUPT, StringType()),
])

### Read orders into bronze

The DataFrame is cached before it is filtered: Spark refuses a query on a raw CSV or JSON file
that references only the corrupt-record column, and caching also stops the file from being
parsed twice for the valid and rejected branches.

In [ ]:
def read_bronze_orders(paths: list[str], batch_id: str):
    """Return (valid, rejected) bronze orders for one batch of landed CSV files."""
    raw = (
        spark.read.schema(ORDERS_SCHEMA)
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", CORRUPT)
        .option("timestampFormat", TS_FORMAT)
        .csv(paths)
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("batch_id", F.lit(batch_id))
        .cache()
    )
    reason = (
        F.when(F.col(CORRUPT).isNotNull(), "unparseable row")
        .when(F.col("order_id").isNull(), "missing order_id")
        .when(F.col("amount") < 0, "negative amount")
        .when(F.col("amount").isNull(), "missing amount")
        .when(F.col("customer_id").isNull(), "missing customer_id")
        .when(F.col("order_ts").isNull() | F.col("updated_at").isNull(), "missing timestamp")
        .when(F.length(F.trim(F.col("status"))).isNull() | (F.trim(F.col("status")) == ""), "missing status")
    )
    flagged = raw.withColumn("reject_reason", reason)
    valid = flagged.filter(F.col("reject_reason").isNull()).drop(CORRUPT, "reject_reason")
    rejected = flagged.filter(F.col("reject_reason").isNotNull()).select(
        "batch_id", "source_file", "reject_reason", F.col(CORRUPT).alias("raw_line"), "order_id"
    )
    return valid, rejected


bronze_orders, rejected_orders = read_bronze_orders([landed["orders_day1.csv"]], "day1")
rejected_orders.show(truncate=False)
print("valid bronze orders:", bronze_orders.count())

### Persist the quarantine

The quarantine is partitioned by `batch_id` and written with an overwrite. With dynamic
partition overwrite that replaces only this batch's partition, so re-running a batch never
doubles its rejects.

In [ ]:
QUARANTINE = os.path.join(LAKE, "quarantine", "orders")


def write_quarantine(rejected) -> None:
    """Replace this batch's partition of the quarantine table."""
    rejected.write.mode("overwrite").partitionBy("batch_id").parquet(QUARANTINE)


write_quarantine(rejected_orders)
reasons = sorted(r["reject_reason"] for r in spark.read.parquet(QUARANTINE).collect())
print("quarantined:", reasons)

In [ ]:
print("landed rows:", len(RAW_FILES["orders_day1.csv"]) - 1, "| valid:", bronze_orders.count(), "| quarantined:", len(reasons))
bronze_orders.select("order_id", "customer_id", "amount", "status", "source_file").orderBy("order_id").show()

**What to notice:** Eleven landed rows split into 7 valid and 4 quarantined, so nothing was lost: two unparseable rows (the
`abc` amount and the `not-a-date` timestamp), one missing key and one negative amount. Bronze still holds
the re-delivered order 2 and both versions of order 3; deduplication is silver's job. Every row carries
its `source_file`.

## Stage 2 - Silver: clean, deduplicate, flatten

Uses: string cleanup ([PySpark 3](pyspark-03-types-functions-and-nulls.ipynb)), `row_number` over a window ([PySpark 7](pyspark-07-aggregations-and-windows.ipynb)) and `explode_outer` on arrays ([PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb)).

Orders keep the **latest version** of each `order_id` by `updated_at`, with `source_file` as a
tie-breaker so the winner is deterministic even for exact duplicates. Customers are trimmed and
upper-cased. Events are deduplicated on `event_id`, and their `items` array is exploded with
`explode_outer` so events with an empty or missing array are not silently dropped.

In [ ]:
from pyspark.sql.window import Window

LATEST_FIRST = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc(), F.col("source_file").desc()
)


def standardize_orders(df):
    """Normalise status text and derive the partition column."""
    return df.withColumn("status", F.lower(F.trim("status"))).withColumn("order_date", F.to_date("order_ts"))


def latest_per_order(df):
    """Keep one row per order_id: the most recent update."""
    return df.withColumn("rn", F.row_number().over(LATEST_FIRST)).filter(F.col("rn") == 1).drop("rn")


def read_silver_customers(path: str):
    """Trimmed names and upper-case ISO country codes."""
    return (
        spark.read.schema(CUSTOMERS_SCHEMA).option("header", True).csv(path)
        .withColumn("name", F.trim("name"))
        .withColumn("country", F.upper(F.trim("country")))
    )


def read_silver_events(paths: list[str]):
    """Return (events, event_items): deduplicated events and one row per item."""
    raw = (
        spark.read.schema(EVENTS_SCHEMA)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", CORRUPT)
        .option("timestampFormat", TS_FORMAT)
        .json(paths)
        .cache()
    )
    events = (
        raw.filter(F.col(CORRUPT).isNull() & F.col("event_id").isNotNull())
        .drop(CORRUPT)
        .dropDuplicates(["event_id"])
    )
    event_items = events.select(
        "event_id", "customer_id", "event_type", F.explode_outer("items").alias("item")
    ).select("event_id", "customer_id", "event_type", "item.sku", "item.qty")
    return events, event_items


silver_day1 = latest_per_order(standardize_orders(bronze_orders))
silver_customers = read_silver_customers(landed["customers.csv"])
silver_events, silver_event_items = read_silver_events([landed["events_day1.jsonl"]])

silver_day1.orderBy("order_id").select("order_id", "customer_id", "amount", "status", "order_date").show()
silver_customers.show()

In [ ]:
silver_events.orderBy("event_id").show(truncate=False)
silver_event_items.orderBy("event_id", "sku").show()

**What to notice:** In silver, order 3 is `delivered` (the later update won), order 2 appears once with its status normalized
to `paid`, and country codes are upper-case. Event `e4` was delivered twice and kept once, and the
truncated `e5` line was dropped. `e3` (empty array) and `e4` (missing array) still have one item row each,
with null SKUs, because of `explode_outer`.

## Stage 3 - Write silver, build gold

Uses: partitioned Parquet writes ([PySpark 4](pyspark-04-reading-and-writing-data.ipynb)), joins ([PySpark 6](pyspark-06-joins.ipynb)), `groupBy`/`pivot` and ranking
windows ([PySpark 7](pyspark-07-aggregations-and-windows.ipynb)).

Silver orders are written partitioned by `order_date`. The write goes through a staging folder
first because Spark refuses to overwrite a path in the same job that reads from it, which the
day-2 upsert in Stage 4 will do. `repartition("order_date")` before the final write gives one
file per date instead of one per task.

Gold is rebuilt from silver on every run. Revenue counts only `paid`, `shipped` and `delivered`
orders, and the customer dimension is joined with an explicit `broadcast` hint and a `left` join,
so an order for an unknown customer lands under `UNKNOWN` instead of disappearing.

In [ ]:
SILVER_ORDERS = os.path.join(LAKE, "silver", "orders")
STAGING_ORDERS = os.path.join(LAKE, "_staging", "orders")
REVENUE_STATUSES = ["paid", "shipped", "delivered"]


def upsert_silver_orders(batch_orders) -> list:
    """Merge a batch into silver, rewriting only the order_date partitions it touches."""
    affected = sorted(r["order_date"] for r in batch_orders.select("order_date").distinct().collect())
    if not affected:
        return []
    merged = batch_orders
    if os.path.exists(SILVER_ORDERS):
        existing = spark.read.parquet(SILVER_ORDERS).filter(F.col("order_date").isin(affected))
        merged = existing.unionByName(batch_orders)
    latest_per_order(merged).write.mode("overwrite").parquet(STAGING_ORDERS)
    (
        spark.read.parquet(STAGING_ORDERS)
        .repartition("order_date")
        .write.mode("overwrite")
        .partitionBy("order_date")
        .parquet(SILVER_ORDERS)
    )
    return affected


def build_gold(orders, customers, events, event_items):
    """Return (revenue_orders, daily_revenue, customer_summary) from silver inputs."""
    revenue_orders = (
        orders.filter(F.col("status").isin(REVENUE_STATUSES))
        .join(F.broadcast(customers.select("customer_id", "country")), "customer_id", "left")
        .withColumn("country", F.coalesce("country", F.lit("UNKNOWN")))
    )
    daily_revenue = revenue_orders.groupBy("order_date", "country").agg(
        F.round(F.sum("amount"), 2).alias("revenue"), F.count("*").alias("orders")
    )
    lifetime = revenue_orders.groupBy("customer_id").agg(
        F.round(F.sum("amount"), 2).alias("lifetime_revenue"), F.count("*").alias("paid_orders")
    )
    activity = events.groupBy("customer_id").pivot("event_type", ["view", "cart"]).count()
    units = (
        event_items.filter(F.col("event_type") == "cart")
        .groupBy("customer_id")
        .agg(F.sum("qty").alias("units_carted"))
    )
    by_country = Window.partitionBy("country").orderBy(F.desc("lifetime_revenue"))
    customer_summary = (
        customers.join(lifetime, "customer_id", "left")
        .join(activity, "customer_id", "left")
        .join(units, "customer_id", "left")
        .na.fill(0, ["lifetime_revenue", "paid_orders", "view", "cart", "units_carted"])
        .withColumn("country_rank", F.dense_rank().over(by_country))
    )
    return revenue_orders, daily_revenue, customer_summary


affected_day1 = upsert_silver_orders(silver_day1)
orders_silver = spark.read.parquet(SILVER_ORDERS)
revenue_orders, daily_revenue, customer_summary = build_gold(
    orders_silver, silver_customers, silver_events, silver_event_items
)
daily_revenue.orderBy("order_date", "country").show()
customer_summary.orderBy("customer_id").show()

In [ ]:
print("silver partitions written:", [str(d) for d in affected_day1])

**What to notice:** Revenue counts only paid, shipped and delivered orders, so customer `c1`'s cancelled order is excluded.
Order 5 belongs to customer `c9`, who is missing from the customer extract; the `left` join keeps its
revenue under `UNKNOWN` instead of losing it. Within each country, `country_rank` orders customers by
lifetime revenue.

## Stage 4 - Incremental re-run on day 2

Uses: incremental loads and idempotent writes ([PySpark 10](pyspark-10-batch-pipelines.ipynb)).

Day 2 brings new orders, a late refund for order 5 (a day-1 date), a **stale** copy of order 3
whose `updated_at` is older than the version silver already holds, and one corrupt row. The
whole pipeline is wrapped in one function and run on it **twice**. A correct incremental job
must:

1. rewrite only the `order_date` partitions the batch touches, leaving `2026-10-01` alone;
2. let the newest `updated_at` win, so the stale order 3 does not regress `delivered` to `shipped`;
3. produce identical silver, gold and quarantine output when the same batch is re-run.

Nothing in the output depends on the wall clock (no `current_timestamp()`), which is what makes
point 3 possible.

In [ ]:
landed["orders_day2.csv"] = land(
    "orders_day2.csv",
    [
        ORDERS_HEADER,
        "5,c9,60.00,refunded,2026-10-03 08:45:00,2026-10-04 09:00:00",
        "3,c3,200.00,shipped,2026-10-02 09:15:00,2026-10-02 09:15:00",
        "9,c4,99.00,paid,2026-10-04 10:00:00,2026-10-04 10:00:00",
        "10,c2,20.00,paid,2026-10-04 11:00:00,2026-10-04 11:00:00",
        "11,c3,12.x,paid,2026-10-04 12:00:00,2026-10-04 12:00:00",
    ],
)


def run_pipeline(order_files: list[str], batch_id: str) -> dict:
    """Run bronze -> silver -> gold for one batch of order files and write every layer."""
    valid, rejected = read_bronze_orders(order_files, batch_id)
    write_quarantine(rejected)
    affected = upsert_silver_orders(standardize_orders(valid))
    _, daily, summary = build_gold(
        spark.read.parquet(SILVER_ORDERS), silver_customers, silver_events, silver_event_items
    )
    daily.write.mode("overwrite").parquet(os.path.join(LAKE, "gold", "daily_revenue"))
    summary.write.mode("overwrite").parquet(os.path.join(LAKE, "gold", "customer_summary"))
    return {"affected": [str(d) for d in affected], "rejected": rejected.count()}


def snapshot(path: str) -> list[tuple]:
    """Sorted rows of a table, for exact before/after comparison."""
    return sorted(tuple(str(v) for v in row) for row in spark.read.parquet(path).collect())


def files_in(partition: str) -> list[str]:
    """Data file names inside one silver partition folder."""
    folder = os.path.join(SILVER_ORDERS, partition)
    return sorted(f for f in os.listdir(folder) if f.endswith(".parquet"))


untouched_before = files_in("order_date=2026-10-01")
first = run_pipeline([landed["orders_day2.csv"]], "day2")
silver_after_first = snapshot(SILVER_ORDERS)
gold_after_first = snapshot(os.path.join(LAKE, "gold", "daily_revenue"))

second = run_pipeline([landed["orders_day2.csv"]], "day2")  # the scheduler retries the same batch
print("first run :", first)
print("second run:", second)
spark.read.parquet(SILVER_ORDERS).orderBy("order_id").select("order_id", "status", "order_date", "batch_id").show()

In [ ]:
print("2026-10-01 files unchanged      :", files_in("order_date=2026-10-01") == untouched_before)
print("silver identical after the retry:", snapshot(SILVER_ORDERS) == silver_after_first)
print("gold identical after the retry  :", snapshot(os.path.join(LAKE, "gold", "daily_revenue")) == gold_after_first)
spark.read.parquet(QUARANTINE).groupBy("batch_id").count().orderBy("batch_id").show()

**What to notice:** Both runs touched the same three dates and rejected the same row, and `2026-10-01` was never rewritten.
Order 5 is now `refunded` (the newer update won), while the stale copy of order 3 did not move it back to
`shipped`. The retry left silver and gold identical, and the quarantine holds four day-1 rejects and one
day-2 reject: rejects were not doubled.

## Stage 5 - Audit the resulting tables

Inspect the published sample tables after both deliveries. Check uniqueness, missing keys and amounts, and allowed statuses. The missing customer `c9` is a known warning. Its order stays in the data and appears under `UNKNOWN` when eligible for revenue.

Compare eligible silver revenue with gold revenue: a count alone misses offsetting errors, so check keys and amounts together, at the same business grain and filter scope. A production job should run its publication checks on the staged result before switching readers to the new version. This local walkthrough audits files already written in the preceding step.

In [ ]:
KNOWN_STATUSES = ["paid", "shipped", "delivered", "cancelled", "refunded"]


def quality_report(orders, customers) -> dict:
    """Count violations of each rule; zero means the rule holds."""
    return {
        "duplicate_order_ids": orders.groupBy("order_id").count().filter(F.col("count") > 1).count(),
        "null_keys": orders.filter(F.col("order_id").isNull() | F.col("customer_id").isNull()).count(),
        "negative_amounts": orders.filter(F.col("amount") < 0).count(),
        "unknown_statuses": orders.filter(F.col("status").isNull() | ~F.col("status").isin(KNOWN_STATUSES)).count(),
        "orphan_customers": orders.join(customers, "customer_id", "left_anti").count(),
    }


orders_silver = spark.read.parquet(SILVER_ORDERS)
report = quality_report(orders_silver, silver_customers)
print(report)

silver_revenue = orders_silver.filter(F.col("status").isin(REVENUE_STATUSES)).agg(F.sum("amount")).first()[0]
gold_revenue = spark.read.parquet(os.path.join(LAKE, "gold", "daily_revenue")).agg(F.sum("revenue")).first()[0]
print("silver revenue:", silver_revenue, "| gold revenue:", gold_revenue)

In [ ]:
hard_rules = {name: count for name, count in report.items() if name != "orphan_customers"}
if any(hard_rules.values()):  # the publication gate: never publish data that breaks a hard rule
    raise ValueError(f"quality gate failed, not publishing: {hard_rules}")
print("quality gate passed | known warning, orphan customers:", report["orphan_customers"])

**What to notice:** Every hard rule reports zero violations, so the gate lets the tables through. The single orphan is customer
`c9`, a known warning rather than a failure. Eligible silver revenue and gold revenue are the same 519.50:
gold reconciles to silver at the same grain and filter.

## Stage 6 - A quick performance check

Uses: reading physical plans, broadcast tuning, partition pruning and file sizing ([PySpark 12](pyspark-12-internals-and-monitoring.ipynb), [PySpark 13](pyspark-13-performance-tuning.ipynb)).

Before shipping, read the plan instead of guessing. Three checks that catch most regressions in
a pipeline like this: the dimension join is a `BroadcastHashJoin` (and falls back to a
`SortMergeJoin` with a shuffle when broadcasting is off), a date filter on silver prunes
partitions instead of scanning them all, and each partition holds one file rather than many
small ones.

In [ ]:
import re


def physical_plan(df) -> str:
    """The physical plan Spark will run, as text."""
    return df._jdf.queryExecution().executedPlan().toString()


revenue_orders, _, _ = build_gold(orders_silver, silver_customers, silver_events, silver_event_items)
hinted_plan = physical_plan(revenue_orders)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
no_broadcast_plan = physical_plan(orders_silver.join(silver_customers, "customer_id", "left"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

one_day = orders_silver.filter(F.col("order_date") == F.lit("2026-10-04").cast("date"))
scan_plan = physical_plan(one_day)
partition_filters = re.search(r"PartitionFilters: \[([^\]]*)\]", scan_plan).group(1)

files_per_partition = {
    folder: len(files_in(folder)) for folder in sorted(os.listdir(SILVER_ORDERS)) if folder.startswith("order_date=")
}
print("hinted join       :", "BroadcastHashJoin" if "BroadcastHashJoin" in hinted_plan else "other")
print("broadcast off     :", "SortMergeJoin" if "SortMergeJoin" in no_broadcast_plan else "other")
print("partition filters :", partition_filters)
print("files per date    :", files_per_partition)

**What to notice:** The hinted dimension join is a `BroadcastHashJoin` under `AdaptiveSparkPlan`; with broadcasting turned off
the same join becomes a `SortMergeJoin` with an `Exchange`. The one-day filter appears in
`PartitionFilters`, so only that date's folder is read, and every date folder holds a single file because
of `repartition("order_date")` before the write.

### Running this on a platform

The logic carries over unchanged; what changes is the storage layer and the scheduler.

| Concern | This notebook | Databricks / Delta Lake | Snowflake |
| --- | --- | --- | --- |
| Landing to bronze | `spark.read` on a folder | Auto Loader with a schema and rescued-data column | `COPY INTO` with `ON_ERROR` and a stage |
| Idempotent upsert | Dynamic partition overwrite | `MERGE INTO` keyed on `order_id` | `MERGE` in a task or dynamic table |
| Small files | `repartition` before write | `OPTIMIZE` and auto-compaction | Managed by the engine |
| Quality gate | `assert` on a report dict | Expectations in a declarative pipeline | Data metric functions or a test task |

## Clean up

Stop the session and remove the temporary lake.

In [ ]:
import shutil

spark.stop()
shutil.rmtree(LAKE, ignore_errors=True)
print("SparkSession stopped and temporary lake removed.")

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Stage 1 caches the raw orders before splitting valid and rejected rows. Why?

**A.** Spark refuses a query on a raw CSV or JSON read that references only the corrupt-record column, and caching also stops the file from being parsed twice for the two branches.

**Q2.** How does the silver dedupe stay deterministic for exact duplicates?

**A.** The window orders by `updated_at` and then by `source_file`, so even two identical rows have a fixed winner on every run.

**Q3.** Why does the stale copy of order 3 on day 2 not overwrite `delivered` with `shipped`?

**A.** The upsert unions existing silver rows for the affected dates with the batch and keeps the newest `updated_at` per order, so an older update loses.

**Q4.** What proves the day-2 rerun is idempotent?

**A.** Silver, gold and the quarantine snapshots are identical after the second run, the untouched `2026-10-01` partition keeps the same files, and rejects are not doubled.

**Q5.** Why is order 5's revenue reported under `UNKNOWN` rather than dropped?

**A.** Gold joins the customer dimension with a `left` join and fills the missing country, so revenue for customer `c9`, absent from the dimension, still reconciles with silver.

**Q6.** What would change to run this on Delta Lake?

**A.** Auto Loader with a rescued-data column for bronze, `MERGE INTO` keyed on `order_id` with a version condition instead of the union-and-overwrite upsert, and `OPTIMIZE` for file sizes.

## Key takeaways

- Declare schemas, quarantine bad rows with a reason, and reconcile landed = valid + rejected.
- Deterministic dedupe needs a source ordering and a tie-breaker.
- Rewrite only the partitions a batch touches, and prove idempotency by running the batch twice.
- Reconcile gold to silver on keys and amounts before publishing.
- Read the plan before shipping: broadcast joins, partition pruning and file counts.

### Next steps

- [PySpark 14: production PySpark](pyspark-14-production-pyspark.ipynb) - pipeline designs, packaging and running jobs.
- [PySpark 15: interview concepts](pyspark-15-interview-concepts.ipynb) - the interview question bank.
- [PySpark coding challenges](pyspark-coding-challenges.ipynb) - interview-style problems with hidden solutions.
- [SQL 8: window functions](../sql/sql-08-window-functions.ipynb) - the same window logic in Spark SQL.